In [1]:
from pathlib import Path
import re
import pandas as pd
from openpyxl import load_workbook

In [2]:
SRC = Path(r"D:\Project\전국도시가스용도별수요가수공급량\data\연도별+회사별+용도별+수요가수+및+공급량(2001_2015).xlsx")
TMP = SRC.with_name(SRC.stem + "_tmp_unmerged.xlsx")

In [14]:
# 1) 병합 해제 저장
wb = load_workbook(SRC, data_only=True)
for ws in wb.worksheets:
    for r in list(ws.merged_cells.ranges):
        ws.unmerge_cells(str(r))
wb.save(TMP)
print("병합 해제 완료 →", TMP)

병합 해제 완료 → D:\Project\전국도시가스용도별수요가수공급량\data\연도별+회사별+용도별+수요가수+및+공급량(2001_2015)_tmp_unmerged.xlsx


In [15]:
# 2) 첫 시트 헤더 없이 읽고, 엑셀 상단 5행(설명/헤더) 제거
sheet_idx = 0
raw = pd.read_excel(TMP, sheet_name=sheet_idx, header=None, dtype=str)
df = raw.iloc[5:].reset_index(drop=True)

# 연도(시트명에서 4자리 숫자)
with pd.ExcelFile(TMP) as xf:
    sheet_name = xf.sheet_names[sheet_idx]
m = re.search(r"(\d{4})", sheet_name)
year = int(m.group(1)) if m else None
print("시트명/연도:", sheet_name, year)

print("전체 열수:", df.shape[1])

시트명/연도: (2001년) 2001
전체 열수: 30


In [16]:
# 3) 왼쪽 2열(시도, 회사) + 나머지 28열을 14/14로 쪼개기
#    - 전제: 현재 총 30열 = 2 + 28(=14·14)
siho = df.iloc[:, :2].copy()     # 시도/회사 원형
rest = df.iloc[:, 2:].copy()     # 나머지 28열

if rest.shape[1] % 2 != 0:
    raise ValueError(f"남은 열이 짝수가 아님: {rest.shape[1]}")

half = rest.shape[1] // 2  # 14
left14  = rest.iloc[:, :half].copy()   # 수요가수 후보
right14 = rest.iloc[:, half:].copy()   # 공급량 후보

# 시도/회사 붙이기
demand_block = pd.concat([siho, left14], axis=1)
supply_block = pd.concat([siho, right14], axis=1)

print("수요가수 블록 shape:", demand_block.shape)
print("공급량 블록 shape:", supply_block.shape)

수요가수 블록 shape: (58, 16)
공급량 블록 shape: (58, 16)


In [17]:
# 4) 열이름 ‘명시’ 지정 (시도/회사 + 14개)
#    집단(있을 수도/없을 수도)을 마지막 자리에 둬서 범용 커버
cols14 = ["가정용","난방용","일반용1","일반용2","소계",
          "업무용","냉난방용","산업용","열병합용","수송용",
          "합계","증감률","구성비","집단"]

demand_block.columns = ["시도","회사"] + cols14
supply_block.columns = ["시도","회사"] + cols14

In [18]:
# 5) 시도/회사 공백 정리 + 아래채우기(ffill)  ← 데이터에는 영향 없음
for col in ["시도","회사"]:
    demand_block[col] = (demand_block[col].astype(str)
                         .str.replace(r"\s+", "", regex=True)
                         .replace({"nan": None})
                         .ffill())
    supply_block[col] = (supply_block[col].astype(str)
                         .str.replace(r"\s+", "", regex=True)
                         .replace({"nan": None})
                         .ffill())

In [19]:
# 6) 숫자형 변환(쉼표/문자 제거 후 to_numeric)
num_cols = [c for c in demand_block.columns if c not in ["시도","회사"]]
for c in num_cols:
    demand_block[c] = (demand_block[c].astype(str)
                       .str.replace(",", "", regex=False)
                       .str.replace(r"[^\d.\-]", "", regex=True)
                       .replace({"": None}))
    demand_block[c] = pd.to_numeric(demand_block[c], errors="coerce")

    supply_block[c] = (supply_block[c].astype(str)
                       .str.replace(",", "", regex=False)
                       .str.replace(r"[^\d.\-]", "", regex=True)
                       .replace({"": None}))
    supply_block[c] = pd.to_numeric(supply_block[c], errors="coerce")

In [20]:
# 7) 파생열 생성: 취사용/업무난방용/개별난방용
for dfb in (demand_block, supply_block):
    dfb["취사용"]     = dfb.get("가정용", 0).fillna(0) - dfb.get("난방용", 0).fillna(0)
    dfb["업무난방용"] = dfb.get("업무용", 0).fillna(0) - dfb.get("냉난방용", 0).fillna(0)
    dfb["개별난방용"] = dfb.get("난방용")  # 이름만 바꾼 개념

In [21]:
# 8) 불필요 열 제거
drop_cols = ["가정용","난방용","업무용","소계","합계","증감률","구성비","집단"]
demand_block = demand_block.drop(columns=[c for c in drop_cols if c in demand_block.columns])
supply_block = supply_block.drop(columns=[c for c in drop_cols if c in supply_block.columns])

In [22]:
# 9) 요약행 제거(소계/지방계/전국계/구성비/주) 등)
rm_pat = r"(소계|지방계|전국계|구성비|\b주\))"
mask_d = demand_block["시도"].fillna("").str.contains(rm_pat) | demand_block["회사"].fillna("").str.contains(rm_pat)
mask_s = supply_block["시도"].fillna("").str.contains(rm_pat) | supply_block["회사"].fillna("").str.contains(rm_pat)

demand_block = demand_block.loc[~mask_d].copy()
supply_block = supply_block.loc[~mask_s].copy()

C:\Users\user\AppData\Local\Temp\ipykernel_19204\898699666.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_d = demand_block["시도"].fillna("").str.contains(rm_pat) | demand_block["회사"].fillna("").str.contains(rm_pat)
C:\Users\user\AppData\Local\Temp\ipykernel_19204\898699666.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_s = supply_block["시도"].fillna("").str.contains(rm_pat) | supply_block["회사"].fillna("").str.contains(rm_pat)


In [23]:
# 10) 최종 컬럼 순서 정리
final_order = ["시도","회사","취사용","개별난방용","일반용1","일반용2",
               "업무난방용","냉난방용","산업용","열병합용","수송용"]
for c in final_order:
    if c not in demand_block.columns: demand_block[c] = pd.NA
    if c not in supply_block.columns: supply_block[c] = pd.NA

demand_block = demand_block[final_order].copy()
supply_block = supply_block[final_order].copy()

In [24]:
# 11) melt → tidy
tidy_demand = demand_block.melt(
    id_vars=["시도","회사"], var_name="용도", value_name="수요가수"
)
tidy_supply = supply_block.melt(
    id_vars=["시도","회사"], var_name="용도", value_name="공급량"
)

if year is not None:
    tidy_demand.insert(0, "연도", year)
    tidy_supply.insert(0, "연도", year)

tidy_demand.head(), tidy_supply.head()

(     연도  시도  회사   용도      수요가수
 0  2001  서울  대한  취사용  183228.0
 1  2001  서울  극동  취사용   82371.0
 2  2001  서울  서울  취사용  209123.0
 3  2001  서울  강남  취사용   39922.0
 4  2001  서울  한진  취사용  170465.0,
      연도  시도  회사   용도       공급량
 0  2001  서울  대한  취사용  -68185.0
 1  2001  서울  극동  취사용  -83172.0
 2  2001  서울  서울  취사용 -127416.0
 3  2001  서울  강남  취사용  -32556.0
 4  2001  서울  한진  취사용  -47716.0)

In [25]:
tidy_demand

,연도,시도,회사,용도,수요가수
0,2001,서울,대한,취사용,183228.0
1,2001,서울,극동,취사용,82371.0
2,2001,서울,서울,취사용,209123.0
3,2001,서울,강남,취사용,39922.0
4,2001,서울,한진,취사용,170465.0
...,...,...,...,...,...
328,2001,경북,포항,수송용,0.0
329,2001,경북,경북,수송용,0.0
330,2001,경북,서라벌,수송용,0.0
331,2001,경남,경남,수송용,0.0
